# ProtoLink: a registry, two agents, and a few simple calls

**One notebook. Ordinary `Agent` instances. Behavior supplied by tools and components.**

We will build a small weather-and-alert system, starting the **registry first**,
then the **WeatherAgent**, then the **AlertAgent**. Along the way, you will use
discovery, call local and remote tools, open the HTTP status/chat pages, inspect
conversation state, and plug in different tools, models, knowledge, and transports.

The application code is just a weather lookup and an alert decision. ProtoLink
already provides the task engine, networking, discovery, and model integration.
There is no agent subclass and no `handle_task` or `call_agent` override.

The default example uses fixture weather and `MockLLM`, so it needs **no API key
or model server**. The tools calculate real results from the fixtures. The mock
chat replies are fixed demonstration text, not generated weather advice. Later,
we show the small change needed to use a real model.

```text
Registry (:9010)       stores the two agents' cards
  ↑          ↑
WeatherAgent (:8010) ← task ← AlertAgent (:8020)
  get_weather                evaluate_weather
```

### Walkthrough

1. [Choose the addresses](#1-choose-the-addresses)
2. [Start the registry](#2-start-the-registry)
3. [Start the WeatherAgent](#3-start-the-weatheragent)
4. [Start the AlertAgent](#4-start-the-alertagent)
5. [Discover and call agents](#5-discover-and-call-agents)
6. [Chat and the HTTP APIs](#6-chat-and-the-http-apis)
7. [Let a model choose an action](#7-let-a-model-choose-an-action)
8. [Plug in more tools](#8-plug-in-more-tools)
9. [Replace a tool implementation](#9-replace-a-tool-implementation)
10. [Add knowledge](#10-add-knowledge)
11. [Inspect conversation state](#11-inspect-conversation-state)
12. [Choose other components](#12-choose-other-components)
13. [Change transports](#13-change-transports)
14. [Stop the services](#14-stop-the-services)

### Open this notebook

Use Python 3.11 or newer. From the repository root:

```bash
python -m pip install -e '.[http,notebook]'
python -m jupyter lab examples/notebooks/basic_example/basic_example.ipynb
```

Select the environment where you installed ProtoLink as the kernel. Run the cells
in order; the notebook uses Jupyter's built-in top-level `await`.

**Run All includes cleanup.** Pause before the last cell to use the browser pages.
Before rerunning startup, run the cleanup cell or restart the kernel. Startup
creates fresh service objects; it is not meant to start the same listener twice.

<a id="1-choose-the-addresses"></a>

## 1. Choose the addresses

These are the only startup settings we need. A transport name tells ProtoLink
which implementation to create. The registry and the agents can use different
transports; for the first run, use HTTP for both so the browser pages are available.

Change a port here if another application is already using it. Section 13 gives
the corresponding names and URL schemes for other transports.

In [ ]:
from protolink import Agent, Task
from protolink.discovery import Registry
from protolink.llms import MockLLM

TRANSPORT = "http"
REGISTRY_TRANSPORT = "http"
REGISTRY_URL = "http://127.0.0.1:9010"
WEATHER_URL = "http://127.0.0.1:8010"
ALERT_URL = "http://127.0.0.1:8020"

<a id="2-start-the-registry"></a>

## 2. Start the registry

The registry holds public agent cards. Agents register themselves, and clients
can then discover their addresses and skills. Task traffic goes directly to the
target agent; the registry is not a task relay.

`background=True` starts the service and returns control to the notebook. We will
pass this registry object directly to both agents.

In [ ]:
registry = Registry(transport=REGISTRY_TRANSPORT, url=REGISTRY_URL, verbosity=0)
registry.start(background=True)

Discovery is already a registry method. At this point, the result is empty because
we have not started either agent.

In [ ]:
await registry.discover()

<a id="3-start-the-weatheragent"></a>

## 3. Start the WeatherAgent

Create an `Agent` with a card, transport, registry, and model. A dictionary is
enough for the card. `registry=registry` reuses the registry's client; there is
no need to construct a second registry or wire a client manually.

The mock model enables the chat interface with a fixed reply. Explicit tool calls
do not need an LLM; they will execute the function we register next.

In [ ]:
weather_agent = Agent(
    card={"name": "WeatherAgent", "description": "Looks up fixture weather.", "url": WEATHER_URL},
    transport=TRANSPORT,
    registry=registry,
    llm=MockLLM(default_response="WeatherAgent is ready. Try the get_weather tool."),
    system_prompt="Use get_weather to answer weather questions about Geneva, Zurich, and Lugano.",
    verbosity=0,
)

### Turn a Python function into an agent tool

The decorator registers the function. Its name, docstring, and annotations supply
the tool metadata and schema. The registered tool is also advertised as a skill
on the agent card.

This is our weather application logic: three fixture readings and one lookup.
Use one of the listed city names when calling this demo tool.

In [ ]:
weather_readings = {
    "Geneva": {"temperature_c": 28, "condition": "sunny"},
    "Zurich": {"temperature_c": 19, "condition": "cloudy"},
    "Lugano": {"temperature_c": 31, "condition": "sunny"},
}


@weather_agent.tool
def get_weather(city: str) -> dict:
    """Look up fixture weather for Geneva, Zurich, or Lugano."""
    return {"city": city, **weather_readings[city]}

Start the agent after attaching its tools. `register=True` publishes its card
to the already-running registry; it is also the default.

In [ ]:
weather_agent.start(register=True, background=True)

Call the tool directly through the public API. `call_tool()` applies the framework's
argument validation and policy checks and returns the function's result.

In [ ]:
await weather_agent.call_tool("get_weather", city="Geneva")

<a id="4-start-the-alertagent"></a>

## 4. Start the AlertAgent

The second agent has a different role and tool. `state=["conversation"]` enables
session history, which we will inspect later. Each agent receives its own model
object.

The system prompt describes how a real model should use the available tools and
peer agent. The simple mock does not interpret that prompt; we will demonstrate
model actions explicitly in section 7.

In [ ]:
alert_agent = Agent(
    card={"name": "AlertAgent", "description": "Evaluates weather alerts.", "url": ALERT_URL},
    transport=TRANSPORT,
    registry=registry,
    llm=MockLLM(default_response="AlertAgent is ready. This is a mock chat reply."),
    system_prompt=(
        "Get weather from WeatherAgent, then use evaluate_weather to decide whether to alert. "
        "Use a threshold of 25 degrees Celsius unless the user asks for another value."
    ),
    state=["conversation"],
    verbosity=0,
)

The alert rule is a single comparison. It returns a decision as data; it does not
send an email or notification. Change `threshold_c` when calling it to use another
threshold. A temperature equal to the threshold does not trigger an alert.

In [ ]:
@alert_agent.tool
def evaluate_weather(city: str, temperature_c: float, threshold_c: float = 25) -> dict:
    """Decide whether a city's temperature exceeds the alert threshold."""
    return {
        "city": city,
        "temperature_c": temperature_c,
        "threshold_c": threshold_c,
        "alert": temperature_c > threshold_c,
    }


alert_agent.start(register=True, background=True)

Now all three services are running. The same tool can produce different decisions
with different arguments; no changes to the agent's task handler are required.

In [ ]:
await alert_agent.call_tool("evaluate_weather", city="Geneva", temperature_c=28)

In [ ]:
await alert_agent.call_tool("evaluate_weather", city="Geneva", temperature_c=28, threshold_c=30)

<a id="5-discover-and-call-agents"></a>

## 5. Discover and call agents

An agent already knows how to query its registry. You can list all agents or filter
by a card field such as `name`. The returned card supplies the address and the
advertised tool schemas.

In [ ]:
await alert_agent.discover_agents()

In [ ]:
weather_cards = await alert_agent.discover_agents({"name": "WeatherAgent"})
weather_card = weather_cards[0]
weather_card.to_dict()

### Send a tool task to the discovered WeatherAgent

`Task.create_tool_call()` packages the tool name and arguments. `call_agent()`
sends it using the configured transport, and the target's built-in task engine
executes the tool. The returned `Task` includes status and a `ToolOutput` containing
the result.

In [ ]:
task = Task.create_tool_call(tool_name="get_weather", args={"city": "Geneva"})
weather_reply = await alert_agent.call_agent(weather_card.url, task)
weather_data = weather_reply.get_last_part_content().result
weather_data

Pass that real result to the AlertAgent's tool. These two calls are the complete
application-directed weather-to-alert workflow.

In [ ]:
await alert_agent.call_tool(
    "evaluate_weather",
    city=weather_data["city"],
    temperature_c=weather_data["temperature_c"],
)

### Choose the amount of control you need

| You want to… | Public API |
|---|---|
| Call a local tool and get its raw result | `await agent.call_tool("tool_name", argument=value)` |
| Ask the current agent's model | `await agent.invoke("Your question")` |
| Execute a local tool as a task | `await agent.invoke("", part_type="tool_call", tool_name="...", tool_args={...})` |
| Send a full task to a peer | `await agent.call_agent(peer_url, task)` |
| Ask a remote agent's model | `await agent.client.send_infer_task("Your question", peer_url)` |
| Inspect a remote agent card | `await agent.client.get_agent_card(peer_url)` |

The existing `agent.client` is an `AgentClient`; use it when you want its additional
operations. An application with no agent of its own can instead construct
`AgentClient(transport="http", url="http://127.0.0.1:0")` as a client-only object.

For debugging a full task, inspect `weather_reply.to_dict()`. In application code,
`weather_reply.raise_for_status()` turns a failed/canceled task into an exception.
Calling the decorated Python function directly is an ordinary function call;
the agent methods are the entry points for framework execution.

In [ ]:
await alert_agent.client.get_agent_card(WEATHER_URL)

<a id="6-chat-and-the-http-apis"></a>

## 6. Chat and the HTTP APIs

The HTTP transport exposes the status pages, agent cards, chat, and task APIs
automatically. We did not define a web application or any routes.

Open these links while the services are running. `GET /status` is an HTML overview;
`GET /.well-known/agent.json` is the structured card. `GET /chat` is the built-in
chat page. The default mock returns the same acknowledgement for each message.

With a real model plugged in, the same page submits questions to that model through
the agent's normal inference loop. Chat is exposed by default; use
`Agent(..., expose_chat=False)` to disable chat interaction. The chat POST route is
mounted when an LLM is present at startup.

In [ ]:
from IPython.display import Markdown, display

display(
    Markdown(f"""
| Service | Status | Chat | Card |
|---|---|---|---|
| Registry | [Open]({REGISTRY_URL}/status) | — | — |
| WeatherAgent | [Open]({WEATHER_URL}/status) | [Open]({WEATHER_URL}/chat) | [JSON]({WEATHER_URL}/.well-known/agent.json) |
| AlertAgent | [Open]({ALERT_URL}/status) | [Open]({ALERT_URL}/chat) | [JSON]({ALERT_URL}/.well-known/agent.json) |
""")
)

From Python, asking your local agent is one call:

In [ ]:
await alert_agent.invoke("Hello!", session_id="notebook-chat")

To ask the remote WeatherAgent, use the existing client's convenience method:

In [ ]:
reply = await alert_agent.client.send_infer_task("Hello!", WEATHER_URL)
reply.get_last_part_content()

### Calling the HTTP endpoints without ProtoLink

An external application can use ordinary JSON requests too. These two short
examples use `httpx` directly just to show the wire API. The previous agent/client
calls already handle HTTP for you.

**HTTP/SSE only:** when experimenting with runtime, WebSocket, or gRPC agents,
skip the next two code cells and use the ProtoLink calls above. Browser pages also
require an HTTP/SSE address.

In [ ]:
import httpx

httpx.get(f"{ALERT_URL}/readyz").json()

In [ ]:
httpx.post(
    f"{ALERT_URL}/chat",
    json={"message": "Hello from the HTTP API!", "session_id": "http-chat"},
).json()

The chat response contains `response`, or `error` when the request cannot be
handled. Conversation history is selected by `session_id`; it persists when the
agent has `state=["conversation"]`. The browser page and these examples can use
different sessions.

### Endpoint reference

| Service | Method and path | What it exposes |
|---|---|---|
| Registry | `GET /agents/` | Registered cards; optional query such as `?name=WeatherAgent` |
| Registry | `POST /agents/` | Register/update a card sent as JSON |
| Registry | `DELETE /agents/` | Unregister with `{"agent_url": "..."}` |
| Registry | `POST /agents/heartbeat` | Refresh liveness with the same `agent_url` body |
| Both | `GET /status` | Human-readable HTML overview |
| Both | `GET /healthz`, `GET /readyz` | Transport health, readiness, capabilities, and metrics |
| Agent | `GET /.well-known/agent.json` | Agent identity and advertised skills |
| Agent | `GET /chat`, `POST /chat` | Chat page and message API |
| Agent | `POST /tasks/` | Send `task.to_dict()`; receive an updated task |
| Agent | `POST /state/describe` | Inspect saved state for a session |
| Agent | `POST /state/compact`, `POST /state/reset` | Compact/reset selected session state |
| Agent | `POST /llm/history/compact` | Compact LLM history |
| Agent | `POST /tasks/cancel` | Request cancellation using `{"id": "task-id"}` |
| Streaming agent | `POST /tasks/stream` | Stream task events when the transport supports it |

`/status` returns HTML, not a JSON status object. `/healthz` and `/readyz` currently
share the transport health implementation; they report whether the transport is
running. Plain HTTP does not stream; the SSE transport adds `/tasks/stream` while
retaining the HTTP pages. This notebook uses ProtoLink's native protocol; A2A
compatibility is a separate opt-in configuration.

<a id="7-let-a-model-choose-an-action"></a>

## 7. Let a model choose an action

So far, Python has chosen which tool or agent to call. During inference, a model
can make that choice instead. We can demonstrate it without writing a custom
model callback: `MockLLM` accepts a sequence of responses.

The first response requests a real call to the registered **WeatherAgent**. The
second is a fixed acknowledgement. ProtoLink resolves the agent name, sends the
tool task, and feeds the result into the inference loop. The acknowledgement is
scripted; it does not summarize or reason about the returned data.

Run the model assignment and invocation together each time you try this example;
a sequence advances as it is consumed. A real model supplies its own responses
instead of this sequence.

In [ ]:
chat_llm = alert_agent.llm
alert_agent.llm = MockLLM(
    sequential_responses=[
        {
            "type": "agent_call",
            "agent": "WeatherAgent",
            "action": "tool_call",
            "tool": "get_weather",
            "args": {"city": "Geneva"},
        },
        "Demo complete: I requested Geneva weather from WeatherAgent.",
    ]
)

await alert_agent.invoke("Ask WeatherAgent for Geneva weather.", session_id="delegation-demo")

Restore the original chat model after the experiment. Swapping an existing model
does not require a new agent class or another server.

In [ ]:
alert_agent.llm = chat_llm

<a id="8-plug-in-more-tools"></a>

## 8. Plug in more tools

Tools all enter through the same registration API. You have already used a
decorator; here is a ready-made built-in tool. Registering it makes it available
immediately, and `register()` publishes the updated card to discovery.

In [ ]:
from protolink.tools import calculator

weather_agent.add_tool(calculator())
await weather_agent.register()
await weather_agent.call_tool("calculator", expression="28 * 9 / 5 + 32")

### Reuse a Python function on another agent

The decorated `evaluate_weather` is still a normal Python function, so it can also
be passed to another agent. The two agents can share a tool implementation while
keeping their own identity, state, model, and other tools.

In [ ]:
weather_agent.add_tool(evaluate_weather)
await weather_agent.register()
await weather_agent.call_tool("evaluate_weather", city="Zurich", temperature_c=19)

For explicit metadata, use a reusable `Tool` object:

```python
from protolink import Tool

tool = Tool.from_callable(evaluate_weather, tags=["weather", "alerts"])
weather_agent.add_tool(tool)
```

These are three forms of the same API: `@agent.tool`, `add_tool(function)`, and
`add_tool(tool_object)`. Existing compatible tool wrappers can be attached in the
same way. The mock model's fixed reply does not automatically start choosing a
new tool; explicit calls work immediately, while a real model can consider the
advertised descriptions and schemas.

<a id="9-replace-a-tool-implementation"></a>

## 9. Replace a tool implementation

The public tool name and argument schema form a small contract. Replace the
implementation under the same name and existing task callers keep working.

Here is an alternate weather provider with a lower temperature. A real adapter
could call an existing weather client and return these same fields. The caller
still asks for `get_weather(city=...)`.

In [ ]:
from protolink import Tool


def cooler_weather(city: str) -> dict:
    """Return an alternate weather fixture."""
    return {"city": city, "temperature_c": 18, "condition": "cloudy"}


original_tool = weather_agent.tools["get_weather"]
weather_agent.add_tool(Tool.from_callable(cooler_weather, name="get_weather"))
await weather_agent.register()

In [ ]:
reply = await alert_agent.call_agent(
    WEATHER_URL,
    Task.create_tool_call(tool_name="get_weather", args={"city": "Geneva"}),
)
reply.get_last_part_content().result

The returned temperature is now 18°C. Restore the original with the same method:

In [ ]:
weather_agent.add_tool(original_tool)
await weather_agent.register()

<a id="10-add-knowledge"></a>

## 10. Add knowledge

Knowledge is another component you can attach. A source becomes a search tool
named `search_<source-name>`, so it can be called locally, over the transport, or
by a model. This in-memory source needs no embedding service or API key.

The two short documents below describe our fictional fixture sensors. We inspect
the actual search result directly; no custom retrieval or response parser is
needed.

In [ ]:
from protolink import Document, create_knowledge

knowledge = create_knowledge(
    "memory",
    name="weather_notes",
    sources=[
        Document(text="The Geneva fixture uses the lakeside demo sensor.", source="geneva.md"),
        Document(text="The Zurich fixture uses the city-center demo sensor.", source="zurich.md"),
    ],
)

weather_agent.add_knowledge(knowledge)
await weather_agent.register()

In [ ]:
await weather_agent.call_tool("search_weather_notes", query="Geneva lakeside sensor", k=1)

You can also provide `knowledge=knowledge` in the agent constructor. After plugging
in a real model, `await weather_agent.ask("Which sensor supplies Geneva?")` retrieves
from the attached source before generating an answer. Our current mock returns
fixed text, so it would not synthesize a grounded answer from those passages.

To repeat the attachment step, first rebuild the agent; knowledge source names
are unique within an agent. Calls to the registered search tool can be repeated
as often as you like.

<a id="11-inspect-conversation-state"></a>

## 11. Inspect conversation state

We enabled `state=["conversation"]` on AlertAgent at construction. A `session_id`
separates one conversation from another. In-memory storage is the default; no
storage adapter was needed to get started.

These messages are saved under different sessions. The mock still returns its
fixed reply, so we inspect the actual stored history instead of pretending it can
recall earlier facts.

In [ ]:
await alert_agent.invoke("My city is Geneva.", session_id="alice")
await alert_agent.invoke("My city is Zurich.", session_id="bob")

The client already has state methods. This is a request to the running agent,
using the same transport as task calls:

In [ ]:
await alert_agent.client.describe_state(ALERT_URL, session_id="alice", include_data=True)

In [ ]:
await alert_agent.client.describe_state(ALERT_URL, session_id="bob", include_data=True)

Compact or reset one session with the corresponding methods. `recent` keeps a
bounded recent history without asking a model to summarize it. Resetting Alice's
session leaves Bob's session alone.

In [ ]:
await alert_agent.client.compact_state(ALERT_URL, session_id="alice", strategy="recent", max_messages=2)

In [ ]:
await alert_agent.client.reset_state(ALERT_URL, session_id="alice", stores=["conversation"])

The lower-level LLM history operation is available as
`await alert_agent.client.compact_history(ALERT_URL, session_id="bob", strategy="recent", max_messages=2)`.

For the full task API, a session belongs on `task.metadata["session_id"]`.
`invoke(..., session_id=...)` is the shorter form when you only need the answer.

<a id="12-choose-other-components"></a>

## 12. Choose other components

The following are small **optional recipes**. Replace the relevant constructor
argument or model assignment when you want to try them; they are not additional
services that Run All starts.

### A real model

Assign a model adapter or pass it as `llm=` when constructing the agent:

```python
from protolink import create_llm

alert_agent.llm = create_llm(
    "ollama", model="your-installed-model", base_url="http://127.0.0.1:11434",
)
await alert_agent.register()
await alert_agent.invoke("Check Geneva weather and decide whether to alert.")
```

Run an Ollama server and choose a model installed there. To use the OpenAI adapter
instead, install the `openai` SDK, set `OPENAI_API_KEY` in the kernel environment,
and use `create_llm("openai", model="your-model")`. Real provider calls use that
provider's resources. The agent tools, task handling, registry, and chat page use
the same interfaces; you replace the model component.

Both agents started with a model, so their chat POST routes already exist. If you
start a different agent with `llm=None`, explicit tool calls still work; build a
fresh server with an LLM when you want to expose its chat POST route.

### Persistent conversation storage

Add these arguments to the AlertAgent constructor:

```python
from protolink.storage import SQLiteStorage

storage=SQLiteStorage(db_path="weather-sessions.db", namespace="alerts"),
state=["conversation"],
```

Reopen the same database and namespace and reuse a session ID to continue that
session across runs. `state=None` makes the agent stateless. Storage selection
and conversation persistence are separate settings; switching storage does not
automatically migrate existing data.

### Save task snapshots

Attach a run store through the constructor:

```python
from protolink import SQLiteRunStore

run_store=SQLiteRunStore("weather-tasks.db"),
```

The run store records task snapshots, while session storage holds conversation
state. Direct `call_tool()` returns a tool result without creating a task; use
`invoke()`, `run_task()`, or a remote task when you want task execution recorded.

### Logging, tracing, and policies

| Component | Plug it in with | What it changes |
|---|---|---|
| Application logger | `logger=` or `verbosity=` | Where/how much diagnostic output appears |
| Local or hosted tracing | `telemetry=` | Observability of task, model, and tool execution |
| Action policy | `policy=` | Runtime authorization of tool and other actions |
| Retriever/knowledge | `knowledge=` | Sources the agent can search |
| Tool integration | `add_tool(...)` | Actions the agent can perform |

For example, `telemetry=LocalTraceTelemetry()` attaches the built-in local tracer
after importing it from `protolink`. These components are constructor options;
none require replacing the agent's execution methods.

<a id="13-change-transports"></a>

## 13. Change transports

The tool functions and task calls do not depend on HTTP. Change `TRANSPORT` and
the two agent URLs in the first cell, then rebuild the agents. Change
`REGISTRY_TRANSPORT` and its URL if you also want to move discovery to that protocol.

| Transport name | Example agent URL | Browser status/chat | Task streams |
|---|---|---|---|
| `http` | `http://127.0.0.1:8010` | Yes | No |
| `sse-json-rpc` | `http://127.0.0.1:8010` | Yes | Yes |
| `websocket` | `ws://127.0.0.1:8010` | No | Yes |
| `runtime` | `runtime://weather` | No | Yes |
| `grpc` | `grpc://127.0.0.1:8010` | No | Yes |

### Run everything in this process

Replace the first cell's settings with:

```python
TRANSPORT = "runtime"
REGISTRY_TRANSPORT = "runtime"
REGISTRY_URL = "runtime://registry"
WEATHER_URL = "runtime://weather"
ALERT_URL = "runtime://alerts"
```

There are no network listeners in this configuration. The same `discover_agents()`,
`call_agent()`, and client state methods work. Skip the two HTTP-only request cells
and browser links in section 6.

For WebSocket or gRPC, use the matching name and scheme for each service you move.
Install `python -m pip install -e '.[grpc]'` before choosing gRPC. An HTTP registry
can still advertise WebSocket agents: keep its own transport and URL as HTTP.
The two agents in this example should share an agent transport so their clients
can call one another.

### Pass an instance when you need options

The name is enough for defaults. For options such as the HTTP backend, replace
an agent constructor's transport argument with an instance:

```python
from protolink.transport import HTTPTransport

transport=HTTPTransport(url=WEATHER_URL, backend="fastapi"),
```

Likewise, `Registry(transport=HTTPTransport(url=REGISTRY_URL))` uses a configured
registry transport. For agents in another process, `registry="http",
registry_url=REGISTRY_URL` is the short form instead of passing this notebook's
registry object.

### Stream a task

With a streaming transport selected, use the existing client:

```python
task = Task.create_tool_call(tool_name="get_weather", args={"city": "Geneva"})
async for event in alert_agent.client.send_task_streaming(WEATHER_URL, task):
    print(event)
```

For SSE, this uses `POST /tasks/stream`. You receive task lifecycle/result events;
the fixed mock does not simulate gradual token generation.

### Cancel a task already in progress

When an application has a task in flight, it can issue:

```python
await alert_agent.client.cancel_task(WEATHER_URL, task.id, reason="No longer needed")
```

Cancellation targets an active task and is cooperative. The fast fixture tools in
this notebook normally finish before a separate cancel request would arrive.
No custom cancellation machinery is required on the agent.

Before changing transport, use the cleanup cell below and then rerun from the top.
Use new service objects; assigning a transport on a running agent does not migrate
the existing listener or its advertised URL.

<a id="14-stop-the-services"></a>

## 14. Stop the services

This is a good place to pause and try the browser chat/status links or make more
tool calls. Once finished, stop the agents before the registry.

The notebook uses `asyncio.to_thread()` only for these blocking shutdown methods,
leaving Jupyter's event loop free while the transports close their connections.
The registry client is shared, so we stop the registry last.

To repeat the walkthrough, run cleanup and then create fresh objects by rerunning
from the top. If startup failed before all three objects existed, restart the
kernel before trying again. A port conflict can be resolved by choosing another
port in the settings; no extra lifecycle framework is needed in the example.

In [ ]:
alert_agent.stop()
weather_agent.stop()
registry.stop()